In [2]:
import os
import random
import pandas as pd
import matplotlib.pyplot as plt
import wfdb

# 1. Configuration
# Change this to where your main mimic demo "files" folder sits
MIMIC_DEMO_DIR = "../projectii/patients" 
OUTPUT_DB_DIR = "./data_ingestion/multi_patient_db"
os.makedirs(OUTPUT_DB_DIR, exist_ok=True)

# Pre-defined variations of medical findings to make each patient unique
DISEASES = ["Congestive Heart Failure", "Acute Pneumonia", "Chronic Hypertension", "Myocardial Ischemia"]
XRAY_FINDINGS = [
    "Enlarged cardiac silhouette with mild bilateral pleural effusions.",
    "Patchy airspace opacities in the right lower lobe, consistent with infection.",
    "Clear lung fields. Normal cardiac size. No acute pulmonary disease.",
    "Mild pulmonary vascular congestion with horizontal lines at the lung bases."
]

def scan_and_generate():
    generated_count = 0
    
    # 2. Iterate through the patient folders (e.g., p10001217)
    for p_folder in os.listdir(MIMIC_DEMO_DIR):
        p_path = os.path.join(MIMIC_DEMO_DIR, p_folder)
        if not os.path.isdir(p_path) or not p_folder.startswith('p'):
            continue
            
        # Look inside for the study subfolder (e.g., s102172660)
        s_folders = [f for f in os.listdir(p_path) if f.startswith('s')]
        if not s_folders:
            continue
        s_folder = s_folders[0]
        
        # Get the actual record name inside the study folder
        study_path = os.path.join(p_path, s_folder)
        record_files = [f for f in os.listdir(study_path) if f.endswith('.hea')]
        if not record_files:
            continue
        record_name = record_files[0].replace('.hea', '')
        full_record_path = os.path.join(study_path, record_name)
        
        # Set up destination
        patient_id = p_folder.replace('p', '')
        dest_patient_dir = os.path.join(OUTPUT_DB_DIR, f"patient_{patient_id}")
        os.makedirs(dest_patient_dir, exist_ok=True)
        
        try:
            # 3. Read real ECG Signal
            signals, fields = wfdb.rdsamp(full_record_path)
            num_leads = fields['n_sig']
            
            # Plot ECG and save it
            plt.figure(figsize=(10, 5))
            for i in range(min(3, num_leads)):
                plt.subplot(3, 1, i + 1)
                plt.plot(signals[:2500, i], color='darkblue', linewidth=0.8)
                plt.grid(True, color='pink', linestyle='--', linewidth=0.5)
            plt.tight_layout()
            plt.savefig(os.path.join(dest_patient_dir, "ecg_plot.jpg"), dpi=100)
            plt.close()
            
            # 4. Generate Matched Tabular Data (Labs)
            disease_idx = generated_count % len(DISEASES)
            wbc = random.choice([14.5, 6.2, 11.1, 8.5])
            bnp = random.choice([4500, 120, 2100, 95])
            
            mock_labs = {
                'Test_Name': ['White Blood Cell (WBC)', 'Hemoglobin', 'NT-proBNP', 'Troponin-T'],
                'Result': [wbc, 13.5, bnp, random.choice([0.15, 0.01, 0.03, 0.45])],
                'Unit': ['x10^3/uL', 'g/dL', 'pg/mL', 'ng/mL'],
                'Flag': ['HIGH' if wbc > 11.0 else 'NORMAL', 'NORMAL', 'HIGH' if bnp > 300 else 'NORMAL', 'HIGH']
            }
            pd.DataFrame(mock_labs).to_csv(os.path.join(dest_patient_dir, "lab_results.csv"), index=False)
            
            # 5. Generate Text Admission Note
            note_text = f"""Patient ID: {patient_id}
            Clinical Presentation: Patient admitted presenting signs highly indicative of {DISEASES[disease_idx]}.
            Diagnostic Studies: A 12-lead ECG waveform was captured for study registry {record_name}.
            Imaging Findings: Chest radiograph completed. Analysis indicates: {XRAY_FINDINGS[disease_idx]}
            Labs: Chemistry panels processed and saved to tabular history."""
            
            with open(os.path.join(dest_patient_dir, "admission_note.txt"), "w") as f:
                f.write(note_text)
                
            # 6. Copy over a placeholder chest X-ray
            # (In production, replace this with a random image from an open dataset)
            # For now, we reuse your ECG image or any dummy jpg just to satisfy the vision pipeline
            plt.figure(figsize=(4,4))
            plt.text(0.5, 0.5, f"Chest X-Ray\nPatient: {patient_id}\n{DISEASES[disease_idx]}", ha='center', va='center')
            plt.savefig(os.path.join(dest_patient_dir, "chest_xray.jpg"))
            plt.close()
            
            print(f"Successfully generated database profile for Patient {patient_id}")
            generated_count += 1
            if generated_count >= 10: # Stop after 10 unique patients
                break
                
        except Exception as e:
            print(f"Skipping record {record_name} due to error: {e}")

if __name__ == "__main__":
    scan_and_generate()
    print("\nDatabase generation complete! You now have a 10-patient multimodal data vault.")


FileNotFoundError: [Errno 2] No such file or directory: '../projectii/patients'

In [3]:
import os
from datasets import load_dataset

DB_DIR = "./data_ingestion/multi_patient_db"

patient_disease_map = {
    "10001217": "Pneumonia",         # # 1. Acute Pneumonia
    "10002428": "Infiltration",      # # 2. Myocardial Ischemia 
    "10004733": "No Finding",        # # 3. Chronic Hypertension 
    "10006053": "Infiltration",      # # 4. Myocardial Ischemia 
    "10007058": "Cardiomegaly",      # # 5. Congestive Heart Failure 
    "10008454": "No Finding",        # # 6. Chronic Hypertension 
    "10009628": "Cardiomegaly",      # # 7. Congestive Heart Failure
    "10010867": "Pneumonia",         # # 8. Acute Pneumonia
    "10011398": "Cardiomegaly",      # # 9. Congestive Heart Failure
    "10013049": "Pneumonia"          # # 10. Acute Pneumonia
}

print("Connecting to NIH Medical Stream on Hugging Face...")
# Stream the dataset instantly without downloading the full archive to your disk
dataset = load_dataset("BahaaEldin0/NIH-Chest-Xray-14", split="train", streaming=True)

# Build a search index map for our required categories
needed_findings = set(patient_disease_map.values())
captured_images = {}

print(f"Searching for real chest radiographs matching: {needed_findings}")
for sample in dataset:
    finding = sample["label"]
    
    for target in needed_findings:
        if target in finding and target not in captured_images:
            captured_images[target] = sample["image"]
            print(f" -> Found a valid clinical image for category: [{target}]")
            
    if len(captured_images) == len(needed_findings):
        break

print("\nInjecting realistic clinical images into your multi-patient directory...")

# Iterate through your 10 active patient folders
for folder in os.listdir(DB_DIR):
    if not folder.startswith("patient_"):
        continue
        
    patient_id = folder.replace("patient_", "")
    
    if patient_id in patient_disease_map:
        target_finding = patient_disease_map[patient_id]
        target_folder_path = os.path.join(DB_DIR, folder)
        xray_output_path = os.path.join(target_folder_path, "chest_xray.jpg")
        
        # Get the distinct medical image asset matching this condition
        raw_pil_image = captured_images[target_finding]
        
        # Overwrite the text-box placeholder with real anatomical data
        raw_pil_image.convert("RGB").save(xray_output_path, "JPEG", quality=90)
        print(f" ✅ Overwrote placeholder: {folder}/chest_xray.jpg with real [{target_finding}] asset.")

print("\nSuccess! Your RAG directory contains genuine, clinically distinct vision layers.")


Connecting to NIH Medical Stream on Hugging Face...


Resolving data files:   0%|          | 0/73 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/73 [00:00<?, ?it/s]

Searching for real chest radiographs matching: {'No Finding', 'Pneumonia', 'Infiltration', 'Cardiomegaly'}
 -> Found a valid clinical image for category: [No Finding]
 -> Found a valid clinical image for category: [Pneumonia]
 -> Found a valid clinical image for category: [Infiltration]
 -> Found a valid clinical image for category: [Cardiomegaly]

Injecting realistic clinical images into your multi-patient directory...
 ✅ Overwrote placeholder: patient_10001217/chest_xray.jpg with real [Pneumonia] asset.
 ✅ Overwrote placeholder: patient_10006053/chest_xray.jpg with real [Infiltration] asset.
 ✅ Overwrote placeholder: patient_10010867/chest_xray.jpg with real [Pneumonia] asset.
 ✅ Overwrote placeholder: patient_10004733/chest_xray.jpg with real [No Finding] asset.
 ✅ Overwrote placeholder: patient_10011398/chest_xray.jpg with real [Cardiomegaly] asset.
 ✅ Overwrote placeholder: patient_10007058/chest_xray.jpg with real [Cardiomegaly] asset.
 ✅ Overwrote placeholder: patient_10002428/c

In [4]:
# The project's README outlines a strategy for overcoming healthcare data silos by implementing a specialized, four-modality ingestion pipeline to construct 10 Multimodal Patient Case Profiles. This approach integrates ECG signals from PhysioNet, anatomical imaging from open archives, tabular lab data modeled on MIMIC-IV schema, and generated audio consultations to ensure genuine clinical reasoning. You can integrate this structured data methodology directly into your README.md to demonstrate robust data alignment skills